# secp256k1_batch — GPU/CPU public-key generation (grouped-affine)

Generates the public keys for a contiguous private-key range using the
**grouped-affine** algorithm (the same technique as VanitySearch / KeyHunt):
each thread generates a block of keys around an affine centre, batch-inverting
the point deltas once per block, so every key costs ~2-3 field
multiplications instead of a full scalar multiplication or a Jacobian add.

Extra optimizations on top:
- **PTX inline-asm field multiply** (`mul.lo`/`mul.hi`/`add.cc`/`addc.cc`, like
  KeyHunt's `GPUMath.h`).
- **Addition-chain inversion** (Brian Smith's chain for `p-2`, ~2.2x faster
  than Fermat) and **symmetric squaring** (10 muls instead of 16).
- **Comb-method setup**: no per-thread scalar multiplication — each thread's
  centre is ~16 point additions from a 32-point table.

1. **Runtime → Change runtime type → GPU** (T4 or better).
2. Run the cells top to bottom; the bootstrap cell clones the repo to
   `/content/secp256k1_batch` and force-resets it to GitHub `main`.
3. Run the cells below.

In [ ]:
# Confirm the GPU is visible.
!nvidia-smi

In [ ]:
# Install CuPy for the GPU backend (bundles the CUDA runtime; no nvcc needed).
# Skips the install if a working CuPy is already present, and clears any
# conflicting CuPy build (e.g. cupy-cuda13x preinstalled on newer images).
import subprocess, sys, importlib

def _cupy_ready():
    try:
        import cupy
        return int(cupy.cuda.runtime.getDeviceCount()) > 0
    except Exception:
        return False

if not _cupy_ready():
    try:
        import torch
        cuda = (torch.version.cuda or "").split(".")[0]
    except Exception:
        cuda = ""
    pkg = f"cupy-cuda{cuda}x" if cuda in ("11", "12", "13") else "cupy-cuda12x"
    for name in ("cupy", "cupy-cuda11x", "cupy-cuda12x", "cupy-cuda13x"):
        subprocess.run([sys.executable, "-m", "pip", "uninstall", "-q", "-y", name], check=False)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=False)
    importlib.invalidate_caches()

In [ ]:
# Bootstrap: always land on the latest GitHub HEAD.  If the folder is not a
# git checkout it is re-cloned; otherwise it is force-reset to origin/main, so
# a failed/partial `git pull` can never leave stale kernel sources behind.
import os, sys, subprocess, importlib, shutil

REPO = "/content/secp256k1_batch"
URL = "https://github.com/huycuonglengoc/secp256k1_batch.git"

if os.getcwd().startswith(REPO):
    os.chdir("/content")

def _git(*args):
    return subprocess.run(["git", "-C", REPO, *args], capture_output=True, text=True)

if os.path.isdir(os.path.join(REPO, ".git")):
    _git("fetch", "-q", "origin")
    _git("reset", "--hard", "-q", "origin/main")
else:
    shutil.rmtree(REPO, ignore_errors=True)
    subprocess.run(["git", "clone", "-q", URL, REPO], check=True)

os.chdir(REPO)
if REPO not in sys.path:
    sys.path.insert(0, REPO)
for _m in [m for m in sys.modules if m == "secp256k1_batch" or m.startswith("secp256k1_batch.")]:
    del sys.modules[_m]
importlib.invalidate_caches()

head = _git("rev-parse", "--short", "HEAD").stdout.strip()
field = open(os.path.join(REPO, "secp256k1_batch/csrc/secp256k1_field.h")).read()
gpupy = open(os.path.join(REPO, "secp256k1_batch/gpu.py")).read()
assert "#define SECP_FIELD_VERSION 4" in field, "stale kernel sources: git update failed"
assert "asm volatile" not in field, "stale kernel sources: inline asm still present"
assert "GPU_BUILD_VERSION = 5" in gpupy, "stale gpu.py: git update failed"
print(f"repo HEAD {head} - kernel sources v4, gpu build v5 (portable, no inline asm)")

In [ ]:
import time
from secp256k1_batch import generate_public_keys, iter_keys
from secp256k1_batch import reference as R

# Smoke test: the first keys must match the trusted reference.
data = generate_public_keys(start=1, count=2000, backend="gpu")
assert list(iter_keys(data)) == R.public_keys_range(1, 2000), "GPU output mismatch!"
print("self-test OK: key #1 x =", next(iter_keys(data))[0].to_bytes(32, "big").hex())

In [ ]:
# Raw GPU kernel throughput (compute only, excludes device->host copy).
from secp256k1_batch.gpu import GPUBackend

be = GPUBackend()  # warm: compiles the kernel + builds the generator table

for count in (1_000_000, 10_000_000, 50_000_000):
    t = be.benchmark_kernel(start=1, count=count)
    print(f"kernel: {count:>12,} keys in {t*1e3:7.2f} ms = {count/t/1e6:7.2f} M keys/s")

# End-to-end (includes the device->host transfer; for large counts this is
# bounded by PCIe bandwidth, not the kernel).
count = 1_000_000
generate_public_keys(start=1, count=128, backend="gpu")  # warm
t = time.perf_counter()
data = generate_public_keys(start=1, count=count, backend="gpu")
dt = time.perf_counter() - t
print(f"end-to-end: {count:,} keys in {dt:.3f}s = {count/dt/1e6:.2f} M keys/s ({len(data)} bytes)")

In [ ]:
# Compressed 33-byte SEC keys (parity prefix + x-coordinate).
data = generate_public_keys(start=1000, count=100_000, compressed=True, backend="gpu")
print(f"compressed bytes: {len(data)} (expected {100_000*33})")

## Tuning

* `grp_size` (default 128): keys per thread per step. Larger amortizes the
  single field inversion over more keys (less work/key) but uses more local
  memory (at 1024 the per-thread arrays spill to DRAM and slow down).
  Sweep 64 / 128 / 256 / 512 / 1024 / 2048.
* `gpu_threads` (default 32768): target grid size.
* CPU-only: `backend="cpu", nthreads=<cores>`.

In [ ]:
# Sweep grp_size on a large count (kernel compute only).
from secp256k1_batch.gpu import GPUBackend

count = 10_000_000
for grp in (64, 128, 256, 512, 1024, 2048):
    be = GPUBackend(grp_size=grp)
    t = be.benchmark_kernel(start=1, count=count)
    print(f"grp_size={grp:5d}: {count/t/1e6:7.2f} M keys/s")